In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("marketing_campaign.csv",sep='\t')

In [ ]:
print(df.shape)
print(df.columns)

In [ ]:
#Finding Columns having null values
df.columns[df.isnull().any()]

In [ ]:
df["Income"].isnull().sum()

In [ ]:
imputer = KNNImputer(n_neighbors=5)
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

print(df['Income'].isnull().sum())

In [ ]:
print(f"Min Birth Year : {df['Year_Birth'].min()}")
print(f"Max Birth Year : {df['Year_Birth'].max()}")

In [ ]:
df["Age"] = 2026 - df["Year_Birth"]
df["Age"].max()

In [ ]:
df["Education"].unique()    

In [ ]:
mappings = {
    'Basic' : 1,
    'Graduation' : 2,
    'Master' : 3,
    '2n Cycle' : 3,  
    'PhD' : 4
}

In [ ]:
df["Education"] = df["Education"].map(mappings)
print(df["Education"].value_counts())

In [ ]:
print(df['Marital_Status'].unique())
print(df["Marital_Status"].value_counts())

In [ ]:
marital_grouping = {
    'Married': 'Partner',
    'Together': 'Partner',
    'Single': 'Single',
    'Divorced': 'Single',
    'Widow': 'Single',
    'Alone': 'Single',
    'Absurd': 'Single',
    'YOLO': 'Single'
}
df['Marital_Status'] = df['Marital_Status'].replace(marital_grouping)

bin_map = {
    'Partner': 1,
    'Single': 0
}
df['Marital_Status'] = df['Marital_Status'].map(bin_map)


In [ ]:
df["Marital_Status"].value_counts()

In [ ]:
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y')
last_date = df['Dt_Customer'].max()
df['Loyalty_Time'] = (last_date - df['Dt_Customer']).dt.days
df = df.drop(columns=['Dt_Customer'])
print(df[['Loyalty_Time']].head())

In [ ]:
df["Z_CostContact"].value_counts()

In [ ]:
df["Z_CostContact"].value_counts()

In [ ]:
for col in df.columns:
    print(f"{col} : {df[col].value_counts()}")

In [ ]:
cols_to_drop = ['ID', 'Year_Birth', 'Z_CostContact', 'Z_Revenue']
df = df.drop(columns=cols_to_drop)

Outlier Removal process

In [ ]:
print("Mean : ",df["Age"].mean())
print("Median : ",df["Age"].median())
print("Std-dev : ",df["Age"].std())
print((df["Age"]>=85).sum())

In [ ]:
plt.hist(df["Age"],bins=100)
plt.plot()

In [ ]:
df = df[df["Age"]<85]

In [ ]:
print("Mean : ",df["Income"].mean())
print("Median : ",df["Income"].median())
print("Std-dev : ",df["Income"].std())
print((df["Income"]>=100000).sum())

In [ ]:
highest_incomes = df[df['Income'] >= 100000]['Income'].sort_values(ascending=False)
print(highest_incomes)

In [ ]:
plt.hist(df["Income"],bins=300)
plt.plot()

In [ ]:
df = df[df['Income'] < 200000]

In [ ]:
df['Total_Spend'] = (df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] + 
                     df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds'])

df['Share_Wine'] = df['MntWines'] / (df['Total_Spend'] + 1e-6)
df['Share_Meat'] = df['MntMeatProducts'] / (df['Total_Spend'] + 1e-6)
df['Share_Fish'] = df['MntFishProducts'] / (df['Total_Spend'] + 1e-6)
df['Share_Sweet'] = df['MntSweetProducts'] / (df['Total_Spend'] + 1e-6)
df['Share_Gold'] = df['MntGoldProds'] / (df['Total_Spend'] + 1e-6)
df['Share_Fruit'] = df['MntFruits'] / (df['Total_Spend'] + 1e-6)

df['Total_Purchases'] = (df['NumWebPurchases'] + df['NumCatalogPurchases'] + df['NumStorePurchases'])

df['Share_Web_Purchases'] = df['NumWebPurchases'] / (df['Total_Purchases'] + 1e-6)
df['Share_Catalog_Purchases'] = df['NumCatalogPurchases'] / (df['Total_Purchases'] + 1e-6)
df['Share_Store_Purchases'] = df['NumStorePurchases'] / (df['Total_Purchases'] + 1e-6)

df['Total_Promos_Accepted'] = (df['AcceptedCmp1'] + df['AcceptedCmp2'] + df['AcceptedCmp3'] + 
                               df['AcceptedCmp4'] + df['AcceptedCmp5'] + df['Response'])

cols_to_drop_agg = [
    'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds',
    'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases',
    'AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'Response'
]

df = df.drop(columns=cols_to_drop_agg)

print("Shape : ", df.shape)

In [ ]:
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df)
df_scaled = pd.DataFrame(scaled_data, columns=df.columns)

df_scaled.to_csv('dataset_cleaned.csv', index=False)

print("Final Shape:", df_scaled.shape)